# Assignment 09: Stacks and queues.


**Specifications and requirements.**  Each assignment must be compliant with the [Programmer's Pact](../housekeeping/ProgrammerPact_Python_2026.pdf).


- You may **not** use the **`seek()`** method in this assignment.
- You may **not** use **lists** for this assignment.
- If your work requires additional **classes,** you may write them.<br/><br/>
- You may **not** import any modules **except** for those already included in the codebase of the assignment or listed below:
  - `from __future__ import annotations`
- No sets or dictionaries may be used.
- If your work requires additional methods to support the development of the methods the assignment asks for, you may write them.

For this assignment, you'll implement a stack and a queue object whose underlying data structure is ... **a file!**

In our examples so far we have built stacks and queue using lists and linked lists as the underlying structures. Now we want to have stack and queue operations using files.

The stack and queue you'll implement will store strings -- one string per line. The Abstract Data Types for your stacks and queues can be found at the bottom of this notebook. The contracts are similar to those we discussed in class:

* **Queue:** `enqueue`, `dequeue`, `peek`, `is_empty`, `size`.
* **Stack:** `push`, `pop`, `peek`, `is_empty`, `size`.

## What to submit

A file called `week09.py` uploaded on Sakai. If your work is done in a Jupyter Notebook, make the notebook shareable and provide its link as the sole line of your `week09.py` file.

## Reading

- [Stacks and Queues](https://learning.oreilly.com/library/view/hands-on-data-structures/9781801073448/Text/Chapter_5.xhtml#_idParaDest-111) from *Hands-On Data Structures and Algorithms with Python* (free from the O'Reilly platform with your LUC email.)

- [Stacks and Queues](https://learning.oreilly.com/library/view/data-structures-the/9781098156602/c04.xhtml#h1-502604c04-0001) from *Data Structures the Fun Way* (free from the O'Reilly platform with your LUC email.)

- [Reading and Writing Files](https://docs.python.org/3/tutorial/inputoutput.html#reading-and-writing-files) from the official Python documentation.

- [Files](https://learning.oreilly.com/library/view/introducing-python-3rd/9781098174392/ch20.html) from Bill Lubanovic's book, (free from the O'Reilly platform with your LUC email).


---

# Brief Tutorial on Stacks & Queues in Python


*Tutorial content adapted from OpenDSA Data Structures & Algorithms (opendsa-server.cs.vt.edu). Rewritten for Python 3 with expanded explanations and examples, by Leo Irakliotis. © OpenDSA Project Contributors, MIT License.*

## Contents

1. [Core Concepts](#1-core-concepts)
2. [Stacks — LIFO](#2-stacks--lifo)
3. [Queues — FIFO](#3-queues--fifo)
4. [Python's Built-in Support](#4-pythons-built-in-support)
5. [Time & Space Complexity](#5-time--space-complexity)
6. [Real-world Applications](#6-real-world-applications)



## 1. Core Concepts

Stacks and queues are *restricted-access* linear data structures. Unlike a plain list where any element can be accessed by index, these structures deliberately limit where insertions and removals may occur. This restriction is a feature, not a limitation: it makes them faster for specific tasks and easier to reason about.

Both structures share a common interface pattern — push/enqueue an element in, pop/dequeue an element out — but they differ in the *order* in which elements come back out.

|                  | **Stack**                        | **Queue**                              |
|------------------|----------------------------------|----------------------------------------|
| Insert/remove at | The **same end** (the top)       | **Rear** to insert, **front** to remove |
| Order            | **Last-In, First-Out** (LIFO)    | **First-In, First-Out** (FIFO)         |
| Mental model     | A stack of plates                | A ticket counter line                  |
| Core operations  | `push`, `pop`, `peek`            | `enqueue`, `dequeue`, `peek`           |
  
  
### ADT vs. Implementation

An *Abstract Data Type (ADT)* defines *what* operations a structure supports and how it behaves — without specifying *how* it's built. A stack ADT says "last in, first out." Whether you use a Python list, a linked list, or a fixed array underneath is an implementation detail.



## 2. Stacks — LIFO

### Concept & Terminology

A stack operates exactly like a pile of books. You can place a book on top (**push**), and you can take the top book off (**pop**). You cannot, without dismantling the pile, reach a book buried in the middle. This is LIFO: the *last* item pushed is the *first* item popped.

```
┌─────────────┐
│  42  ← top  │   ← most recently pushed
├─────────────┤
│     17      │
├─────────────┤
│      5      │
├─────────────┤
│     99      │   ← first pushed
└─────────────┘
  Stack state after pushing 99, 5, 17, 42
```

Key vocabulary:

- **top** — the accessible element (most recently pushed)
- **push(item)** — add an item to the top
- **pop()** — remove and return the top item
- **peek()** — inspect the top item without removing it
- **is_empty()** — check whether the stack has no elements


### Implementation 1 — Using a Python List

Python's built-in `list` is a dynamic array. It supports O(1) amortised appends and pops from the right end, making the right end a natural "top" of the stack. This is the idiomatic Python approach.

```python
class Stack:
    """Array-backed stack using Python list as the underlying store."""

    def __init__(self):
        self._data = []          # right end of list = top of stack

    def push(self, item):
        """O(1) amortised — append to the right end."""
        self._data.append(item)

    def pop(self):
        """O(1) — remove and return the top element."""
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self._data.pop()

    def peek(self):
        """O(1) — inspect without removing."""
        if self.is_empty():
            raise IndexError("peek at empty stack")
        return self._data[-1]

    def is_empty(self):
        return len(self._data) == 0

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"Stack({self._data} ← top)"


# ── Demo ──────────────────────────────────────────
s = Stack()
for val in [99, 5, 17, 42]:
    s.push(val)

print(s)          # Stack([99, 5, 17, 42] ← top)
print(s.peek())   # 42
print(s.pop())    # 42
print(s.pop())    # 17
print(len(s))     # 2
```

### Why the right end?

Python's `list.append()` and `list.pop()` (no argument) both operate on the rightmost element in O(1) amortised time. `list.pop(0)` — removing from the left — is O(n) because every remaining element must shift. Always use the right end as your stack top.



### Implementation 2 — Fixed-Capacity Array Stack

When you know the maximum number of elements in advance, a fixed-capacity stack uses a pre-allocated array and an integer `top` index. This avoids any memory reallocation overhead and is a direct mirror of how stacks are taught in lower-level languages. The design decision: which end of the array is the top? Using the *right* end (high index) keeps push and pop at O(1).

```python
class FixedStack:
    """Stack backed by a fixed-size list. Raises if capacity is exceeded."""

    def __init__(self, capacity: int = 10):
        self._capacity = capacity
        self._data = [None] * capacity   # pre-allocate
        self._top = -1                   # -1 means empty

    def push(self, item):
        if self._top >= self._capacity - 1:
            raise OverflowError("stack is full")
        self._top += 1
        self._data[self._top] = item

    def pop(self):
        if self.is_empty():
            raise IndexError("pop from empty stack")
        item = self._data[self._top]
        self._data[self._top] = None   # allow GC to reclaim
        self._top -= 1
        return item

    def peek(self):
        if self.is_empty():
            raise IndexError("peek at empty stack")
        return self._data[self._top]

    def is_empty(self):
        return self._top == -1

    def __len__(self):
        return self._top + 1

    def clear(self):
        self._data = [None] * self._capacity
        self._top = -1
```

### Tracing a Push and Pop

Walk through what happens when we push three values then pop two:

```python
fs = FixedStack(capacity=5)

fs.push(10)   # _data: [10, None, None, None, None]   _top: 0
fs.push(20)   # _data: [10,  20,  None, None, None]   _top: 1
fs.push(30)   # _data: [10,  20,   30,  None, None]   _top: 2

fs.pop()      # returns 30  — _top: 1
fs.pop()      # returns 20  — _top: 0
fs.peek()     # returns 10  — _top unchanged: 0
```

> ⚠️ **Null out stale references**
>
> After a pop, setting the freed slot to `None` lets Python's garbage collector reclaim objects that are no longer logically in the stack. Skipping this step can cause subtle memory leaks when storing large objects.



## 3. Queues — FIFO

### Concept & Terminology

A queue works like a line at a coffee shop: newcomers join at the back, and the person at the front is served next. Elements exit in the same order they arrived — First-In, First-Out (FIFO).

```
  dequeue ←                          → enqueue
     ┌────────┬────────┬────────┬────────┐
     │   7    │   12   │    3   │   88   │
     │ front  │        │        │  rear  │
     └────────┴────────┴────────┴────────┘
```

Key vocabulary:

- **front** — the next element to be removed
- **rear** — where the next element will be inserted
- **enqueue(item)** — add an item to the rear
- **dequeue()** — remove and return the front item
- **peek()** — inspect the front item without removing it



### Naïve List Approach — and Why It Fails

A first instinct is to use `list.append()` to enqueue at the right and `list.pop(0)` to dequeue from the left. This works correctly but is **O(n) per dequeue**: every remaining element must shift one position left. For a queue processing millions of items this is a performance disaster.

```python
# Functionally correct — but O(n) dequeue! Don't do this.
queue = []
queue.append(1)         # enqueue
queue.append(2)
first = queue.pop(0)    # dequeue — shifts ALL elements left → O(n)
```



### Implementation 1 — Circular Buffer (Array-Based)

The classic solution to the O(n) shifting problem is a *circular buffer*. Rather than shifting elements, we advance the `front` and `rear` pointers using modular arithmetic. Conceptually, the array wraps around on itself like a ring — when a pointer reaches the end, it "circles back" to index 0.

**The full/empty ambiguity:** With an array of size `n`, both a full queue and an empty queue can produce `front == rear`, making them indistinguishable. The standard fix is to allocate `n + 1` slots and only ever allow `n` elements — leaving one slot deliberately empty as a sentinel.

```python
class CircularQueue:
    """
    Fixed-capacity queue using a circular buffer.
    Allocates (capacity + 1) slots to distinguish full from empty.
    All core operations run in O(1).
    """

    def __init__(self, capacity: int = 10):
        self._size  = capacity + 1          # extra sentinel slot
        self._data  = [None] * self._size
        self._front = 0                     # index of next element to dequeue
        self._rear  = self._size - 1        # index of most recently enqueued

    # ── Internal helpers ─────────────────────────────────────────

    def _increment(self, idx):
        """Advance an index by 1 with wrap-around."""
        return (idx + 1) % self._size

    # ── Public interface ──────────────────────────────────────────

    def enqueue(self, item):
        """Add item to the rear of the queue. O(1)."""
        if self.is_full():
            raise OverflowError("queue is full")
        self._rear = self._increment(self._rear)
        self._data[self._rear] = item

    def dequeue(self):
        """Remove and return the front element. O(1)."""
        if self.is_empty():
            raise IndexError("dequeue from empty queue")
        item = self._data[self._front]
        self._data[self._front] = None      # release reference
        self._front = self._increment(self._front)
        return item

    def peek(self):
        """Return the front element without removing it. O(1)."""
        if self.is_empty():
            raise IndexError("peek at empty queue")
        return self._data[self._front]

    def is_empty(self):
        # rear is one step behind front when empty
        return self._increment(self._rear) == self._front

    def is_full(self):
        # rear is two steps behind front when full (sentinel in between)
        return self._increment(self._increment(self._rear)) == self._front

    def __len__(self):
        return (self._rear - self._front + self._size + 1) % self._size

    def clear(self):
        self._data  = [None] * self._size
        self._front = 0
        self._rear  = self._size - 1
```

### Tracing the Circular Buffer

Let's trace a 4-element capacity queue (5 slots allocated). Watch how the pointers advance without any shifting:

```python
q = CircularQueue(capacity=4)
# Initial:  data=[None,None,None,None,None]  front=0  rear=4

q.enqueue('A')
# rear → 0   data=['A', None, None, None, None]  front=0
q.enqueue('B')
# rear → 1   data=['A',  'B', None, None, None]  front=0
q.enqueue('C')
# rear → 2   data=['A',  'B',  'C', None, None]  front=0

q.dequeue()    # returns 'A'   front → 1
# data=[None, 'B', 'C', None, None]  front=1  rear=2

q.enqueue('D')
# rear → 3   data=[None, 'B', 'C', 'D', None]  front=1
q.enqueue('E')
# rear wraps → 4  data=[None,'B','C','D','E']   front=1

q.dequeue()    # returns 'B'   front → 2
```

> **Visualising the wrap-around**
>
> The key insight: when `rear` reaches the last index, the next enqueue sets `rear = (rear + 1) % size = 0`, looping back to the beginning. The data never moves — only the pointers advance. This gives O(1) for both enqueue and dequeue.



## 4. Python's Built-in Support

In practice, Python ships with highly optimised data structures that should be your first choice unless you need to implement from scratch for learning purposes.

### Stack — just use a list

Python's built-in `list` is already a perfect stack. No wrapper class needed:

```python
stack = []

stack.append(10)   # push
stack.append(20)   # push
stack.append(30)   # push

stack[-1]          # peek → 30
stack.pop()        # pop  → 30
stack.pop()        # pop  → 20
```

### Queue — use `collections.deque`

`collections.deque` (double-ended queue) is implemented as a doubly-linked list of fixed-size blocks. It provides O(1) appends and pops from *both ends*, making it the correct Python tool for queues.

```python
from collections import deque

queue = deque()

queue.append('first')    # enqueue at right
queue.append('second')
queue.append('third')

queue[0]                 # peek → 'first'
queue.popleft()          # dequeue → 'first'
queue.popleft()          # dequeue → 'second'
```


## 5. Time & Space Complexity

All basic stack and queue operations, when implemented correctly, run in constant time. This is what makes them attractive: no matter how many items are stored, each individual operation takes the same amount of work.



## 6. Real-world Applications

**Stacks in practice:**
- Function call stack (how Python tracks recursion)
- Undo / redo in editors
- Bracket / parenthesis matching
- Expression evaluation (infix → postfix)
- Browser back-button history
- Depth-First Search (DFS)

**Queues in practice:**
- CPU process scheduling
- Print spoolers
- Web server request handling
- Message brokers (Kafka, RabbitMQ)
- Breadth-First Search (BFS)
- Buffering I/O streams



### Example — Balanced Brackets (Stack)

A classic stack application: determine whether a string's brackets are properly balanced and nested.

```python
def is_balanced(text: str) -> bool:
    """
    Return True if every opening bracket in `text` is matched
    by the corresponding closing bracket in the correct order.
    Uses a stack to track unmatched openers.
    """
    matching = {')': '(', ']': '[', '}': '{'}
    stack = []

    for ch in text:
        if ch in '([{':
            stack.append(ch)                  # push opener
        elif ch in ')]}':
            if not stack or stack[-1] != matching[ch]:
                return False                  # unmatched closer
            stack.pop()                       # matched — discard opener

    return not stack                          # True iff all openers were matched


print(is_balanced("({[]})"))   # True
print(is_balanced("([)]"))     # False — wrong nesting order
print(is_balanced("((("))      # False — unclosed openers remain
```





---



In [7]:
from abc import ABC, abstractmethod
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Type, Union


class QueueADT(ABC):
    @abstractmethod
    def enqueue(self, item: Any) -> None:
        pass

    @abstractmethod
    def dequeue(self) -> Any:
        pass

    @abstractmethod
    def is_empty(self) -> bool:
        pass

    @abstractmethod
    def size(self) -> int:
        pass

    @abstractmethod
    def peek(self) -> Any:
        pass

In [8]:
from abc import ABC, abstractmethod
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Type, Union

class StackADT(ABC):
    @abstractmethod
    def push(self, item: Any) -> None:
        pass

    @abstractmethod
    def pop(self) -> Any:
        pass

    @abstractmethod
    def peek(self) -> Any:
        pass

    @abstractmethod
    def is_empty(self) -> bool:
        pass

    @abstractmethod
    def size(self) -> int:
        pass